<a href="https://colab.research.google.com/github/WuilderOsio1198/Senalesysistemas/blob/main/TALLER2/TALLER2PRIMERAPARTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Ejercicio 1.1
Cuaderno 1
- Implemente un filtro pasa bajas, un pasa altas, un pasa bandas, y un rechaza bandas utilizando la FFT y la iFFT sobre 5 segundos de su canción favorita de YouTube.



In [ ]:
!python3 -m pip install --force-reinstall https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz -q

In [ ]:
#Cargue las cookies del navegador para permitir la autenticación y el acceso al contenido autorizado.
from google.colab import files
import shutil, os

print("Selecciona el archivo .txt de cookies exportado desde tu navegador (ej: youtube.com_cookies.txt).")
uploaded = files.upload()

if not uploaded:
    print("No se subió ningún archivo.")
else:
    # toma el primer archivo subido
    original_name = next(iter(uploaded))
    dest = "/content/cookies_yt.txt"
    # si ya existe, lo sobreescribimos
    if os.path.exists(dest):
        os.remove(dest)
    shutil.move(original_name, dest)
    print(f"Archivo subido y guardado como: {dest}")
    print("Contenido (primeras 5 líneas) — para confirmar formato:")
    with open(dest, "r", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= 5: break
            print(line.rstrip())

In [ ]:
#Se debe incluir el link del video a procesar:
link="https://www.youtube.com/watch?v=R70wWRqWU_c&list=RDR70wWRqWU_c&start_radio=1"
!yt-dlp --extract-audio -o "audio" --audio-format mp3 --force-overwrites --cookies cookies_yt.txt {link}

In [ ]:
#Se convierte el audio a .wav para procesar:
!ffmpeg -y -i audio.mp3 output.wav

In [ ]:
#Librería para manipulación de archivos de audio
!pip install soundfile

In [ ]:
import soundfile as sf # para instalar pip install soundfile
#lee archivos wav
nombre_out = "output.wav"
x, fs = sf.read(nombre_out)
# read speech signal from file
print('Frecuencia de muestreo %.2f[Hz]\naudio %s' % (fs,nombre_out))

In [ ]:
#El audio viene codificado en dos canales:
x.shape

In [ ]:
from IPython.display import Audio
ns = 30 #se reproducen los primeros ns segundos
Audio(x[:int(fs*ns),:].T,rate=fs)

In [ ]:
#Se gráfica un intervalo de la señal en el tiempo:
import numpy as np # Import numpy
import matplotlib.pyplot as plt # Import matplotlib for plotting

xpro = x.copy() #copiar archivos para procesar
ti = 10 #tiempo incio a procesar seg
tf = 15 #tiempo final a procesar seg
xs = xpro[int(ti*fs):int((tf*fs)),:]

tt = np.arange(ti,tf,1/fs) # vector de tiempo
plt.plot(tt,xs)
plt.legend(('canal 1','canal 2'))
plt.xlabel('$t[s]$')
plt.ylabel('$x(t)$')
plt.legend()
plt.show()

In [ ]:
#Se cálcula el espectro de Fourier del segmento de audio escogido sobre cada canal (sonido estereo).
Xw = np.fft.rfft(xs,axis=0) # axis=0 permite aplicar fft por cada columna de xpro
#Xwc1 = np.fft.rfft(xs[:,0])
#Xwc2 = np.fft.rfft(xs[:,1])
vf = np.fft.rfftfreq(np.size(xs,0),1/fs) #se crea el vector de frecuencias
plt.plot(vf,abs(Xw))#se grafica la magnitud
plt.legend(('canal 1','canal 2'))
plt.title(r'Espectro audio original')
plt.xlabel(r'$f[Hz]$',fontsize = 14)
plt.ylabel(r'$|X[n]|$',fontsize = 14)
plt.show()

In [ ]:
#FILTRO PASABANDAS

#filtrar espectro
Xwf = Xw.copy()
f1 = 100 #frecuencia en Hz corte 1
f2 = 800 #frecuencia en Hz corte 2
ind = ~((vf > f1) & (vf < f2)) #frecuencias eliminar-> recueder que ~ actua como negación
Xwf[ind,:] = 0
plt.plot(vf,abs(Xwf))
plt.legend(('canal 1','canal 2'))
plt.show()

In [ ]:
xe2 = np.fft.irfft(Xwf,axis=0) #fft inversa sobre los dos canales de audio
Audio(xe2[:int(fs*ns),:].T,rate=fs)#repoducir señal filtrada

In [ ]:
# FILTRO PASABAJAS

# Hacemos una copia del espectro para trabajar sobre ella
Xwf = Xw.copy()

# 1. DEFINIR LA FRECUENCIA DE CORTE
f_corte = 400 #Hz

#'ind' marca en True las frecuencias mayores al corte que se eliminan.
ind = abs(vf) > f_corte

# Hacemos cero las componentes del espectro para las frecuencias altas y medias.
Xwf[ind,:] = 0

# Graficamos el espectro ya filtrado.
plt.plot(vf,abs(Xwf))
plt.legend(('canal 1','canal 2'))
plt.show()

In [ ]:
xe2 = np.fft.irfft(Xwf,axis=0) #fft inversa sobre los dos canales de audio
Audio(xe2[:int(fs*ns),:].T,rate=fs)#repoducir señal filtrada

In [ ]:
#FILTRO PASA ALTAS

# Hacemos una copia del espectro para trabajar sobre ella
Xwf = Xw.copy()

# Todas las frecuencias por DEBAJO de este valor serán eliminadas.
f_corte = 1000 # Frecuencia de corte en Hz

# El índice 'ind' será Verdadero (True) para las frecuencias que queremos ELIMINAR. Es decir, todas aquellas cuya magnitud sea MENOR que f_corte.
ind = abs(vf) < f_corte

# Hacemos cero las componentes del espectro para las frecuencias bajas.
Xwf[ind,:] = 0

# Graficamos el espectro ya filtrado.
plt.plot(vf,abs(Xwf))
plt.legend(('canal 1','canal 2'))
plt.show()

In [ ]:
xe2 = np.fft.irfft(Xwf,axis=0) #fft inversa sobre los dos canales de audio
Audio(xe2[:int(fs*ns),:].T,rate=fs)#repoducir señal filtrada

In [ ]:
#FILTRO RECHAZA BANDAS

# Hacemos una copia del espectro para trabajar sobre ella
Xwf = Xw.copy()

# f1 es el inicio de la banda a eliminar y f2 es el final.
f1 = 400  # Frecuencia de corte inferior en Hz
f2 = 1500 # Frecuencia de corte superior en Hz

# El índice 'ind' será Verdadero (True) para las frecuencias dentro de la banda que queremos eliminar.
ind = (abs(vf) > f1) & (abs(vf) < f2)

# 3. APLICAR EL FILTRO
# Hacemos cero las componentes del espectro dentro de la banda rechazada.
Xwf[ind,:] = 0

# Graficamos el espectro ya filtrado.
plt.plot(vf,abs(Xwf))
plt.legend(('canal 1','canal 2'))
plt.show()

In [ ]:
xe2 = np.fft.irfft(Xwf,axis=0) #fft inversa sobre los dos canales de audio
Audio(xe2[:int(fs*ns),:].T,rate=fs)#repoducir señal filtrada

In [ ]:
Fo = 8000
F2 = 16000
Fs = 80000
ti = 0
tf = 5
tv = np.arange(ti,tf,1/Fs)
A = 30
xt = A*np.cos(2*np.pi*Fo*tv) +0.5*A*np.cos(2*np.pi*F2*tv)
plt.plot(tv,xt)
plt.show()

In [ ]:
Audio(xt,rate=Fs)#repoducir señal filtrada

In [ ]:
Xwc = np.fft.rfft(xt) # axis=0 permite aplicar fft por cada columna de xpro
vfc = np.fft.rfftfreq(xt.shape[0],1/Fs)

plt.plot(vfc,abs(Xwc))#se grafica la magnitud
plt.title(r'Espectro cos')
plt.xlabel(r'$f[Hz]$',fontsize = 14)
plt.ylabel(r'$|X[n]|$',fontsize = 14)
plt.show()

In [ ]:
#Guardar archivo filtrado en .wav
name_out_fil = 'new_file.wav'
sf.write(name_out_fil, xe2, fs)
print('Audio filtrado\n')